In [ ]:
import cellestial as cl
import scanpy as sc


from lets_plot import *

import polars as pl

LetsPlot.setup_html()


data = sc.read("data/pbmc3k_pped.h5ad")

In [ ]:
dimensions = "umap"
key = "leiden"
cluster_name = "Cluster"
barcode_name = "CellID"

frame = pl.from_numpy(
    data.obsm[f"X_{dimensions}"][:, :2], schema=[f"{dimensions}1", f"{dimensions}2"]
).with_columns(pl.Series(barcode_name, data.obs_names))


frame = frame.with_columns(
    pl.Series(barcode_name, data.obs_names), pl.Series(cluster_name, data.obs[key])
)

In [ ]:
umap = cl.dimensional(data)

In [ ]:
frame

In [ ]:
grouped = frame.group_by(cluster_name).agg(
    pl.col(f"{dimensions}1").mean(), pl.col(f"{dimensions}2").mean())
grouped

In [ ]:
(
    umap
    + geom_text(
        data=grouped,
        mapping=aes(x=f"{dimensions}1", y=f"{dimensions}2", label=cluster_name),
        size = 12,
        color="#3f3f3f",
        fontface="bold",
        family="sans",
        alpha=0.8,
    )
    + theme(legend_position="none")
    + ggtitle("Aritmetic")
)+ ggsize(700,600)

In [ ]:
cl.show_colors()

        size = 12,
        color="#3f3f3f",
        fontface="bold",
        family="sans",
        alpha=0.4,

addd these as with `ondata_` prefix.
rest is up to a dict `ondata_dict`.
ondata_dict docstring 
should have 
https://lets-plot.org/python/pages/api/lets_plot.geom_text.html

In [ ]:
x = f"{dimensions}1"
y = f"{dimensions}2"
if True:
    group_means = frame.group_by(cluster_name).agg(
        pl.col(x).mean().alias("mean_x"), pl.col(y).mean().alias("mean_y")
    )
    # join the group means to the frame
    frame = frame.join(group_means, on=cluster_name, how="left")
    # calculate the distance between the group means and the frame
    frame = frame.with_columns(
        ((pl.col(x) - pl.col("mean_x")) ** 2 + (pl.col(y) - pl.col("mean_y")) ** 2)
        .sqrt()
        .alias("distance")
    )
    # assign weights to the individual points
    frame = frame.with_columns((1 / pl.col("distance").sqrt()).alias("weight"))
    # calculate the weighted mean of the group means
    grouped2 = frame.group_by(cluster_name).agg(
        (pl.col(x) * pl.col("weight")).sum() / pl.col("weight").sum(),
        (pl.col(y) * pl.col("weight")).sum() / pl.col("weight").sum(),
    )

In [ ]:
grouped2

In [ ]:
(
    umap
    + geom_text(
        data=grouped2,
        mapping=aes(x=f"{dimensions}1", y=f"{dimensions}2", label=cluster_name),
        size = 12,
        color="#3f3f3f",
        fontface="bold",
        family="sans",
        alpha=0.8,
    )
    + theme(legend_position="none")
    + ggtitle("WEIGHTED")

)+ ggsize(700,600)

In [ ]:
grouped.sort("Cluster")

In [ ]:
grouped2.sort("Cluster")

In [ ]:
cl.tsne(data, legend_ondata=True)

In [ ]:
cl.tsne(data, legend_ondata=True,ondata_weighted=False)

In [ ]:
sc.pl.tsne(data, color="leiden",legend_loc="on data")